In [ ]:
# --- bootstrap: make src/ importable and run from the repository root ---
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "path_info.py").exists())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [ ]:
!pip install adjustText

In [ ]:
# dbutils.library.restartPython()  # Databricks only

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn.functional as F
from torchvision import transforms, datasets
from transformers import BlipModel, BlipProcessor
from tqdm import tqdm
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np
import glob
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, classification_report
import re
from functools import lru_cache
import multiprocessing
import mmap
import json

from transformers import BlipModel, BlipProcessor, CLIPModel, CLIPProcessor

from baseline_model import CustomClassifier, plot_history
from CBM_model import CBM_model, plot_concept_to_class_weights
from data_CUB_multimodal import PATH_DATA_CUB, load_data_CUB

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub.file_download")

os.environ["TOKENIZERS_PARALLELISM"] = "false" # to remove warnings about parallelism :
# huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
# To disable this warning, you can either:
# - Avoid using `tokenizers` before the fork if possible
# - Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

# XP Our Best CBM (loss leakage, KAN)

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = 'data/datasets/N24News/combine/cb_llm_annotation/outputs_concept_scoring/blue_checkpoints/clip/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

history, model = run_CBM(dataset='N24', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone='clip', 
                concept_representation='importance', 
                num_epochs=10, 
              #  import_concept_list=concept_scores,
              #  concept_level=12,
                load=False, 
                plot=True, 
                leakage_loss=False, 
                leakage_loss_activation='up', 
                kan_layer=False,
                loss_CBLLM='MSE',
                independant=True)

In [ ]:

import joblib

joblib.dump(model, "saved_models/model_intervention_20260311_5.joblib")

In [ ]:
from pathlib import Path

print(*sorted(p.name for p in Path("saved_models").glob("*.joblib")), sep=chr(10))


In [ ]:
model = joblib.load("saved_models/model_intervention_20260306_2.joblib")

In [ ]:
from data_N24_concepts import load_data_N24, PATH_DATA_N24
from CBM_pipeline import batch_size, max_len

train_loader, val_loader, test_loader, class_dict, concept_list, concept_counts = load_data_N24(data_dir=PATH_DATA_N24, class_list=None, batch_size=batch_size, max_len=max_len, dataset_type='CBLLM', combine_type='combine', select_concepts=[concept.replace('concept_','') for concept in model.concept_list])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
from intervention import CBMInterventionEvaluator

# Create evaluator
evaluator = CBMInterventionEvaluator(model)

# Step 1: Compute intervention ordering on validation set
print("="*70)
print("STEP 1: Computing Intervention Ordering")
print("="*70)
intervention_ordering = evaluator.compute_intervention_ordering(val_loader)

# Step 2: Evaluate interventions on test set
print("\n" + "="*70)
print("STEP 2: Evaluating Test-Time Interventions")
print("="*70)
results = evaluator.evaluate_interventions(test_loader, max_interventions=10)

# Step 3: Plot results
print("\n" + "="*70)
print("STEP 3: Visualizing Results")
print("="*70)
evaluator.plot_intervention_curve(results)

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Baseline accuracy (no intervention): {results['accuracy'][0]:.4f}")
if len(results['accuracy']) > 2:
    print(f"Accuracy with 2 interventions: {results['accuracy'][2]:.4f}")
    print(f"Improvement: {results['accuracy'][2] - results['accuracy'][0]:.4f}")

In [ ]:
results

In [ ]:
import os

os.makedirs("saved_intervention_results", exist_ok=True)

In [ ]:
with open("saved_intervention_results/results_intervention_20260311_5.json", "w") as f:
    json.dump(results, f)

In [ ]:
import json

with open("saved_intervention_results/results_intervention_20260305_1.json", "r") as f:
    results_today = json.load(f)

print(results_today)

In [ ]:
print('test')

# XP Label Free (cos-cubed concept loss, linear final layer)

In [ ]:
from CBM_pipeline import run_CBM

history, model = run_CBM(dataset='N24', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone='clip', 
                concept_representation='importance', 
                num_epochs=10, 
#                import_concept_list=concept_scores,
#                concept_level=12,
                load=False, 
                plot=True, 
                leakage_loss=False, 
                leakage_loss_activation='up', 
                kan_layer=False,
                loss_CBLLM='cos_cubed')

In [ ]:
joblib.dump(model, "saved_models/model_intervention_20260306_2.joblib")

In [ ]:
from data_N24_concepts import load_data_N24, PATH_DATA_N24
from CBM_pipeline import batch_size, max_len

train_loader, val_loader, test_loader, class_dict, concept_list, concept_counts = load_data_N24(data_dir=PATH_DATA_N24, class_list=None, batch_size=batch_size, max_len=max_len, dataset_type='CBLLM', combine_type='combine', select_concepts=[concept.replace('concept_','') for concept in model.concept_list])

In [ ]:
from intervention import CBMInterventionEvaluator

# Create evaluator
evaluator = CBMInterventionEvaluator(model)

# Step 1: Compute intervention ordering on validation set
print("="*70)
print("STEP 1: Computing Intervention Ordering")
print("="*70)
intervention_ordering = evaluator.compute_intervention_ordering(val_loader)

# Step 2: Evaluate interventions on test set
print("\n" + "="*70)
print("STEP 2: Evaluating Test-Time Interventions")
print("="*70)
results = evaluator.evaluate_interventions(test_loader, max_interventions=10)

# Step 3: Plot results
print("\n" + "="*70)
print("STEP 3: Visualizing Results")
print("="*70)
evaluator.plot_intervention_curve(results)

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Baseline accuracy (no intervention): {results['accuracy'][0]:.4f}")
if len(results['accuracy']) > 2:
    print(f"Accuracy with 2 interventions: {results['accuracy'][2]:.4f}")
    print(f"Improvement: {results['accuracy'][2] - results['accuracy'][0]:.4f}")